# 06_silver_validation

## Purpose

Validate the Day 3 Silver layer.

This notebook checks that all expected Silver tables exist, contain rows, expose clean schemas, and satisfy basic quality checks for nulls and invalid values.

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

catalog = "vattenfall_dev"
schema = "refined"

silver_tables = [
    f"{catalog}.{schema}.silver_market_prices",
    f"{catalog}.{schema}.silver_weather",
    f"{catalog}.{schema}.silver_grid_events",
    f"{catalog}.{schema}.silver_asset_reference",
    f"{catalog}.{schema}.silver_regional_operations_base",
]

In [0]:
validation_rows = []

for table_name in silver_tables:
    df = spark.table(table_name)

    validation_rows.append(
        Row(
            table_name=table_name,
            row_count=df.count(),
            column_count=len(df.columns),
            columns=", ".join(df.columns)
        )
    )

validation_summary_df = spark.createDataFrame(validation_rows)

display(validation_summary_df)

In [0]:
empty_tables = validation_summary_df.filter(F.col("row_count") <= 0)

if empty_tables.count() > 0:
    display(empty_tables)
    raise ValueError("Silver validation failed. One or more Silver tables are empty.")

print("All expected Silver tables exist and contain rows.")

In [0]:
for table_name in silver_tables:
    print(f"\nSchema for {table_name}")
    spark.table(table_name).printSchema()

In [0]:
null_checks = {
    f"{catalog}.{schema}.silver_market_prices": [
        "region", "market_type", "price_eur_mwh", "volume_mwh", "report_day"
    ],
    f"{catalog}.{schema}.silver_weather": [
        "region", "temperature_c", "wind_speed_kmh", "precipitation_mm", "weather_alert_level", "report_day"
    ],
    f"{catalog}.{schema}.silver_grid_events": [
        "event_id", "region", "asset_id", "event_type", "severity", "severity_band", "duration_minutes", "event_day"
    ],
    f"{catalog}.{schema}.silver_asset_reference": [
        "asset_id", "asset_name", "region", "asset_type"
    ],
    f"{catalog}.{schema}.silver_regional_operations_base": [
        "event_id", "asset_id", "region", "event_day"
    ],
}

null_check_rows = []

for table_name, columns in null_checks.items():
    df = spark.table(table_name)

    for column_name in columns:
        null_check_rows.append(
            Row(
                table_name=table_name,
                column_name=column_name,
                null_count=df.filter(F.col(column_name).isNull()).count()
            )
        )

null_check_df = spark.createDataFrame(null_check_rows)

display(null_check_df)

In [0]:
invalid_check_rows = []

market_df = spark.table(f"{catalog}.{schema}.silver_market_prices")
weather_df = spark.table(f"{catalog}.{schema}.silver_weather")
grid_df = spark.table(f"{catalog}.{schema}.silver_grid_events")
asset_df = spark.table(f"{catalog}.{schema}.silver_asset_reference")

invalid_check_rows.append(
    Row(
        check_name="market_price_null_or_invalid",
        invalid_count=market_df.filter(
            (F.col("price_eur_mwh").isNull()) |
            (F.col("volume_mwh").isNull())
        ).count()
    )
)

invalid_check_rows.append(
    Row(
        check_name="negative_wind_speed",
        invalid_count=weather_df.filter(F.col("wind_speed_kmh") < 0).count()
    )
)

invalid_check_rows.append(
    Row(
        check_name="negative_grid_event_duration",
        invalid_count=grid_df.filter(F.col("duration_minutes") < 0).count()
    )
)

invalid_check_rows.append(
    Row(
        check_name="duplicate_asset_id",
        invalid_count=(
            asset_df
            .groupBy("asset_id")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )
    )
)

invalid_check_df = spark.createDataFrame(invalid_check_rows)

display(invalid_check_df)

In [0]:
print("Market types:")
market_df.select("market_type").distinct().show(truncate=False)

print("Weather alert levels:")
weather_df.select("weather_alert_level").distinct().show(truncate=False)

print("Grid event severities:")
grid_df.select("severity").distinct().show(truncate=False)

print("Grid event severity bands:")
grid_df.select("severity_band").distinct().show(truncate=False)

print("Asset types:")
asset_df.select("asset_type").distinct().show(truncate=False)

In [0]:
for table_name in silver_tables:
    print(f"\nSample rows for {table_name}")
    display(spark.table(table_name).limit(20))